# 05 — Matrix Multiplication

In the previous notebooks, we learned how to create tensors, manipulate tensor shapes, perform tensor operations, and use broadcasting.

Now we will study one of the most important operations in deep learning:

> **Matrix multiplication**

Matrix multiplication is used throughout deep learning:

- Linear layers
- Neural networks
- Attention mechanisms
- Embeddings
- Feature transformations
- CNN classifiers
- Transformers

## In this notebook, we will learn:

1. Element-wise multiplication vs matrix multiplication
2. Dot product intuition
3. Matrix multiplication rules
4. Inner-dimension rule
5. Output-shape reasoning
6. `torch.matmul()`
7. `@` operator
8. Matrix-vector multiplication
9. Matrix-matrix multiplication
10. Batched matrix multiplication
11. `torch.bmm()`
12. Transpose and matrix multiplication
13. Neural-network linear layers from first principles
14. Common matrix multiplication errors
15. Deep-learning shape exercises

## Main Goal

If:

$$
A.shape=(m,\ n)
$$

and:

$$
B.shape=(n,\ p)
$$

you should immediately recognize:

$$
A @ B \rightarrow (m,\ p)
$$

> **Check the inner dimensions first. Then keep the outer dimensions.**


In [ ]:
import torch

print("PyTorch version:", torch.__version__)


# 1. Element-Wise Multiplication vs Matrix Multiplication

Consider:

$$
A =
\begin{array}{|c|c|}
\hline
1 & 2 \\
\hline
3 & 4 \\
\hline
\end{array}
$$

and:

$$
B =
\begin{array}{|c|c|}
\hline
5 & 6 \\
\hline
7 & 8 \\
\hline
\end{array}
$$

There are two very different multiplication operations:

- `A * B` → element-wise multiplication
- `A @ B` → matrix multiplication


## Element-Wise Multiplication

Element-wise multiplication multiplies values in the same positions:

$$
A * B =
\begin{array}{|c|c|}
\hline
1\times5 & 2\times6 \\
\hline
3\times7 & 4\times8 \\
\hline
\end{array}
$$

Therefore:

$$
A * B =
\begin{array}{|c|c|}
\hline
5 & 12 \\
\hline
21 & 32 \\
\hline
\end{array}
$$


In [ ]:
A = torch.tensor([
    [1, 2],
    [3, 4]
])

B = torch.tensor([
    [5, 6],
    [7, 8]
])

print("A * B:")
print(A * B)


## Matrix Multiplication

Matrix multiplication uses:

> **one row from the first matrix and one column from the second matrix**

$$
A @ B =
\begin{array}{|c|c|}
\hline
1(5)+2(7) & 1(6)+2(8) \\
\hline
3(5)+4(7) & 3(6)+4(8) \\
\hline
\end{array}
$$

Therefore:

$$
A @ B =
\begin{array}{|c|c|}
\hline
19 & 22 \\
\hline
43 & 50 \\
\hline
\end{array}
$$


In [ ]:
print("A @ B:")
print(A @ B)


## Important Difference

$$
\begin{array}{|c|c|}
\hline
\textbf{Operation} & \textbf{Meaning} \\
\hline
A * B & \text{Element-wise multiplication} \\
\hline
A @ B & \text{Matrix multiplication} \\
\hline
torch.mul(A,B) & \text{Element-wise multiplication} \\
\hline
torch.matmul(A,B) & \text{Matrix multiplication} \\
\hline
\end{array}
$$


# 2. Dot Product Intuition

Consider two vectors:

$$
x =
\begin{array}{|c|c|c|}
\hline
1 & 2 & 3 \\
\hline
\end{array}
$$

and:

$$
y =
\begin{array}{|c|c|c|}
\hline
4 & 5 & 6 \\
\hline
\end{array}
$$

The dot product is:

$$
x \cdot y =
(1\times4)+(2\times5)+(3\times6)
$$

$$
=4+10+18
$$

$$
=\boxed{32}
$$


In [ ]:
x = torch.tensor([1, 2, 3])
y = torch.tensor([4, 5, 6])

print("Dot product:", torch.dot(x, y))


## Dot Product in Simple Words

A dot product:

1. Multiplies corresponding values
2. Adds the products
3. Produces one scalar

So:

$$
(3)\cdot(3)\rightarrow()
$$


In [ ]:
products = x * y

print("Element-wise products:", products)
print("Sum of products:", products.sum())


# 3. Matrix Multiplication as Repeated Dot Products

Consider:

$$
A =
\begin{array}{|c|c|c|}
\hline
1 & 2 & 3 \\
\hline
4 & 5 & 6 \\
\hline
\end{array}
$$

and:

$$
B =
\begin{array}{|c|c|}
\hline
7 & 8 \\
\hline
9 & 10 \\
\hline
11 & 12 \\
\hline
\end{array}
$$

Shapes:

$$
A.shape=(2,\ 3)
$$

$$
B.shape=(3,\ 2)
$$

Each output entry is a **row-column dot product**.


## First Output Value

First row of `A`:

$$
\begin{array}{|c|c|c|}
\hline
1 & 2 & 3 \\
\hline
\end{array}
$$

First column of `B`:

$$
\begin{array}{|c|}
\hline
7 \\
\hline
9 \\
\hline
11 \\
\hline
\end{array}
$$

So:

$$
(1\times7)+(2\times9)+(3\times11)=\boxed{58}
$$


In [ ]:
A = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])

B = torch.tensor([
    [7, 8],
    [9, 10],
    [11, 12]
])

C = A @ B

print("A @ B:")
print(C)
print("Shape:", C.shape)


The complete result is:

$$
A @ B =
\begin{array}{|c|c|}
\hline
58 & 64 \\
\hline
139 & 154 \\
\hline
\end{array}
$$


# 4. The Inner-Dimension Rule

Suppose:

$$
A.shape=(m,\ n)
$$

and:

$$
B.shape=(n,\ p)
$$

Matrix multiplication is valid because the **inner dimensions match**:

$$
\boxed{n=n}
$$

The output shape comes from the **outer dimensions**:

$$
A @ B \rightarrow (m,\ p)
$$

A useful pattern is:

$$
(m,\ \boxed{n}) @ (\boxed{n},\ p)
\rightarrow
(m,\ p)
$$


## Example

$$
(3,\ \boxed{4}) @ (\boxed{4},\ 5)
\rightarrow
\boxed{(3,\ 5)}
$$


In [ ]:
A = torch.randn(3, 4)
B = torch.randn(4, 5)

C = A @ B

print("A shape:", A.shape)
print("B shape:", B.shape)
print("C shape:", C.shape)


# 5. Invalid Matrix Multiplication

Suppose:

$$
A.shape=(3,\ 4)
$$

and:

$$
B.shape=(2,\ 5)
$$

The inner dimensions are:

$$
4 \neq 2
$$

Therefore:

$$
(3,\ 4) @ (2,\ 5)
$$

is invalid.


In [ ]:
A = torch.randn(3, 4)
B = torch.randn(2, 5)

print("A shape:", A.shape)
print("B shape:", B.shape)

# Uncomment the line below to see the error:
# C = A @ B


# 6. Output-Shape Reasoning

For:

$$
(a,\ b) @ (c,\ d)
$$

first check:

$$
\boxed{b=c}
$$

If valid:

$$
(a,\ b) @ (b,\ d)
\rightarrow
(a,\ d)
$$


## Examples

$$
(2,\ 3) @ (3,\ 4)
\rightarrow
\boxed{(2,\ 4)}
$$

$$
(10,\ 128) @ (128,\ 64)
\rightarrow
\boxed{(10,\ 64)}
$$

$$
(32,\ 784) @ (784,\ 10)
\rightarrow
\boxed{(32,\ 10)}
$$

The last example is similar to a neural-network classifier.


# 7. `torch.matmul()`

PyTorch provides:

`torch.matmul(A, B)`

for matrix multiplication.


In [ ]:
A = torch.randn(2, 3)
B = torch.randn(3, 4)

C = torch.matmul(A, B)

print("Output shape:", C.shape)


# 8. The `@` Operator

The `@` operator is a shorter way to write matrix multiplication.

These are equivalent:

`torch.matmul(A, B)`

and:

`A @ B`


In [ ]:
C1 = torch.matmul(A, B)
C2 = A @ B

print("Same values:", torch.allclose(C1, C2))


# 9. Matrix-Vector Multiplication

Suppose:

$$
A.shape=(2,\ 3)
$$

and:

$$
x.shape=(3)
$$

Then:

$$
(2,\ 3) @ (3)
\rightarrow
(2)
$$

Example:

$$
A =
\begin{array}{|c|c|c|}
\hline
1 & 2 & 3 \\
\hline
4 & 5 & 6 \\
\hline
\end{array}
$$

$$
x =
\begin{array}{|c|}
\hline
10 \\
\hline
20 \\
\hline
30 \\
\hline
\end{array}
$$


In [ ]:
A = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])

x = torch.tensor([10, 20, 30])

y = A @ x

print("A shape:", A.shape)
print("x shape:", x.shape)
print("y shape:", y.shape)
print("y:", y)


The output is:

$$
\begin{array}{|c|}
\hline
1(10)+2(20)+3(30) \\
\hline
4(10)+5(20)+6(30) \\
\hline
\end{array}
=
\begin{array}{|c|}
\hline
140 \\
\hline
320 \\
\hline
\end{array}
$$


# 10. Matrix-Matrix Multiplication

Suppose:

$$
A.shape=(2,\ 3)
$$

and:

$$
B.shape=(3,\ 4)
$$

Then:

$$
(2,\ 3) @ (3,\ 4)
\rightarrow
\boxed{(2,\ 4)}
$$

The result has:

- 2 rows from `A`
- 4 columns from `B`


In [ ]:
A = torch.randn(2, 3)
B = torch.randn(3, 4)

C = A @ B

print("A shape:", A.shape)
print("B shape:", B.shape)
print("C shape:", C.shape)


# 11. Transpose and Matrix Multiplication

Transpose swaps rows and columns.

Consider:

$$
B =
\begin{array}{|c|c|c|}
\hline
1 & 2 & 3 \\
\hline
4 & 5 & 6 \\
\hline
\end{array}
$$

Shape:

$$
\boxed{(2,\ 3)}
$$

Its transpose is:

$$
B^T =
\begin{array}{|c|c|}
\hline
1 & 4 \\
\hline
2 & 5 \\
\hline
3 & 6 \\
\hline
\end{array}
$$

Shape:

$$
\boxed{(3,\ 2)}
$$


In [ ]:
B = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])

print("B:")
print(B)
print("B shape:", B.shape)

print("\nB.T:")
print(B.T)
print("B.T shape:", B.T.shape)


## Why Transpose Often Appears With Weights

Suppose:

$$
X.shape=(5,\ 3)
$$

and:

$$
W.shape=(4,\ 3)
$$

Direct multiplication:

$$
(5,\ 3) @ (4,\ 3)
$$

does not work.

But:

$$
W^T.shape=(3,\ 4)
$$

so:

$$
(5,\ 3) @ (3,\ 4)
\rightarrow
\boxed{(5,\ 4)}
$$


In [ ]:
X = torch.randn(5, 3)
W = torch.randn(4, 3)

Y = X @ W.T

print("X shape:", X.shape)
print("W shape:", W.shape)
print("W.T shape:", W.T.shape)
print("Y shape:", Y.shape)


# 12. Batched Matrix Multiplication

Deep-learning models often process many matrices at once.

Suppose:

$$
A.shape=(B,\ M,\ N)
$$

and:

$$
C.shape=(B,\ N,\ P)
$$

Then:

$$
A @ C
\rightarrow
(B,\ M,\ P)
$$

The first dimension is the **batch dimension**.


In [ ]:
A = torch.randn(8, 3, 4)
B = torch.randn(8, 4, 5)

C = A @ B

print("A shape:", A.shape)
print("B shape:", B.shape)
print("C shape:", C.shape)


Shape reasoning:

$$
(8,\ 3,\ \boxed{4})
@
(8,\ \boxed{4},\ 5)
\rightarrow
\boxed{(8,\ 3,\ 5)}
$$


# 13. `torch.bmm()`

`torch.bmm()` performs batch matrix multiplication for **3D tensors**.

Required shapes:

$$
(B,\ M,\ N)
$$

and:

$$
(B,\ N,\ P)
$$

Output:

$$
(B,\ M,\ P)
$$


In [ ]:
A = torch.randn(4, 2, 3)
B = torch.randn(4, 3, 5)

C = torch.bmm(A, B)

print("A shape:", A.shape)
print("B shape:", B.shape)
print("C shape:", C.shape)


## `torch.matmul()` vs `torch.bmm()`

$$
\begin{array}{|c|c|}
\hline
\textbf{torch.matmul()} & \textbf{torch.bmm()} \\
\hline
\text{Supports multiple ranks} & \text{Works with 3D batch tensors} \\
\hline
\text{Can broadcast batch dimensions} & \text{Does not broadcast batch dimensions} \\
\hline
\text{Very flexible} & \text{Explicit batch matrix multiplication} \\
\hline
\end{array}
$$


# 14. `torch.matmul()` With Batch Broadcasting

Suppose:

$$
A.shape=(10,\ 3,\ 4)
$$

and:

$$
B.shape=(4,\ 5)
$$

The same `(4,5)` matrix can be multiplied with every matrix in the batch:

$$
(10,\ 3,\ 4) @ (4,\ 5)
\rightarrow
\boxed{(10,\ 3,\ 5)}
$$


In [ ]:
A = torch.randn(10, 3, 4)
B = torch.randn(4, 5)

C = torch.matmul(A, B)

print("A shape:", A.shape)
print("B shape:", B.shape)
print("C shape:", C.shape)


# 15. Neural-Network Linear Layer From First Principles

A linear layer is built from matrix multiplication plus bias.

Suppose:

$$
X.shape=(batch,\ in\_features)
$$

PyTorch stores weights as:

$$
W.shape=(out\_features,\ in\_features)
$$

and bias as:

$$
b.shape=(out\_features)
$$

The operation is:

$$
Y=XW^T+b
$$

Shapes:

$$
(batch,\ in)
@
(in,\ out)
+
(out)
$$

giving:

$$
\boxed{(batch,\ out)}
$$


## Concrete Example

Suppose:

$$
X.shape=(32,\ 784)
$$

and:

$$
W.shape=(10,\ 784)
$$

Then:

$$
W^T.shape=(784,\ 10)
$$

Therefore:

$$
(32,\ 784) @ (784,\ 10)
\rightarrow
\boxed{(32,\ 10)}
$$

The bias of shape `(10)` broadcasts across all 32 samples.


In [ ]:
batch_size = 32
in_features = 784
out_features = 10

X = torch.randn(batch_size, in_features)
W = torch.randn(out_features, in_features)
b = torch.randn(out_features)

Y = X @ W.T + b

print("X shape:", X.shape)
print("W shape:", W.shape)
print("W.T shape:", W.T.shape)
print("b shape:", b.shape)
print("Y shape:", Y.shape)


# 16. Comparing With `torch.nn.Linear`

Now compare the manual operation with PyTorch's `nn.Linear`.


In [ ]:
import torch.nn as nn

X = torch.randn(4, 3)

linear = nn.Linear(in_features=3, out_features=2)

manual_output = X @ linear.weight.T + linear.bias
pytorch_output = linear(X)

print("Weight shape:", linear.weight.shape)
print("Bias shape:", linear.bias.shape)
print("Output shape:", pytorch_output.shape)

print("Outputs match:", torch.allclose(manual_output, pytorch_output))


For:

`nn.Linear(3, 2)`

PyTorch stores:

$$
weight.shape=(2,\ 3)
$$

$$
bias.shape=(2)
$$

For input:

$$
X.shape=(batch,\ 3)
$$

the operation is:

$$
X @ weight^T + bias
$$

so:

$$
(batch,\ 3) @ (3,\ 2)
\rightarrow
(batch,\ 2)
$$


# 17. A Single Neuron From First Principles

A single neuron can be understood as a dot product plus bias.

Suppose:

$$
x =
\begin{array}{|c|c|c|}
\hline
x_1 & x_2 & x_3 \\
\hline
\end{array}
$$

and:

$$
w =
\begin{array}{|c|c|c|}
\hline
w_1 & w_2 & w_3 \\
\hline
\end{array}
$$

Then:

$$
z=x\cdot w+b
$$

or:

$$
z=x_1w_1+x_2w_2+x_3w_3+b
$$


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
w = torch.tensor([0.5, -1.0, 2.0])
b = torch.tensor(0.1)

z = torch.dot(x, w) + b

print("Neuron output:", z)


# 18. Multiple Neurons

If we have multiple neurons, each neuron has its own weight vector.

Stacking the weight vectors creates a weight matrix.

Suppose:

$$
x.shape=(3)
$$

and:

$$
W.shape=(2,\ 3)
$$

Then:

$$
x @ W^T
$$

gives:

$$
(3) @ (3,\ 2)
\rightarrow
\boxed{(2)}
$$


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])

W = torch.tensor([
    [0.5, -1.0, 2.0],
    [1.0,  0.5, -0.5]
])

b = torch.tensor([0.1, -0.2])

output = x @ W.T + b

print("Output:", output)
print("Output shape:", output.shape)


# 19. Deep-Learning Shape Example

Suppose an MNIST batch is flattened into:

$$
X.shape=(64,\ 784)
$$

A hidden layer contains 128 neurons:

$$
W_1.shape=(128,\ 784)
$$

Then:

$$
(64,\ 784) @ (784,\ 128)
\rightarrow
\boxed{(64,\ 128)}
$$

A second layer with 10 output neurons uses:

$$
W_2.shape=(10,\ 128)
$$

Therefore:

$$
(64,\ 128) @ (128,\ 10)
\rightarrow
\boxed{(64,\ 10)}
$$


In [ ]:
X = torch.randn(64, 784)

W1 = torch.randn(128, 784)
b1 = torch.randn(128)

W2 = torch.randn(10, 128)
b2 = torch.randn(10)

hidden = X @ W1.T + b1
output = hidden @ W2.T + b2

print("Input shape:", X.shape)
print("Hidden shape:", hidden.shape)
print("Output shape:", output.shape)


# 20. Matrix Multiplication Is Not Commutative

For ordinary numbers:

$$
a\times b=b\times a
$$

But matrix multiplication generally does not satisfy:

$$
AB=BA
$$

Even when both products are valid, the results may be different.


In [ ]:
A = torch.tensor([
    [1.0, 2.0],
    [3.0, 4.0]
])

B = torch.tensor([
    [0.0, 1.0],
    [2.0, 3.0]
])

print("A @ B:")
print(A @ B)

print("\nB @ A:")
print(B @ A)

print("\nEqual:", torch.allclose(A @ B, B @ A))


# 21. Common Matrix Multiplication Errors

## Mistake 1 — Using `*` Instead of `@`

`A * B` performs element-wise multiplication.

`A @ B` performs matrix multiplication.

## Mistake 2 — Ignoring Inner Dimensions

Always check:

$$
(m,\ \boxed{n}) @ (\boxed{n},\ p)
$$

## Mistake 3 — Transposing Randomly

Do not use `.T` only to remove an error.

First understand what each dimension means.

## Mistake 4 — Losing the Batch Dimension

A batch commonly has shape:

$$
(batch,\ features)
$$

Make sure the batch dimension is preserved.

## Mistake 5 — Confusing Weight Orientation

For:

`nn.Linear(in_features, out_features)`

PyTorch stores:

$$
weight.shape=(out,\ in)
$$

so the manual operation uses:

`X @ weight.T`

## Mistake 6 — Using `torch.bmm()` on Non-3D Tensors

`torch.bmm()` expects two 3D tensors.


# 22. Matrix Multiplication Debugging Checklist

Whenever matrix multiplication fails, inspect:

- `A.shape`
- `B.shape`
- `A.ndim`
- `B.ndim`
- `A.dtype`
- `B.dtype`
- `A.device`
- `B.device`

Then ask:

1. What does each dimension represent?
2. Are the inner dimensions equal?
3. Did I transpose the correct matrix?
4. Am I using element-wise or matrix multiplication?
5. Is there a batch dimension?
6. Should I use `matmul()` or `bmm()`?
7. Are both tensors on the same device?


In [ ]:
A = torch.randn(8, 32)
B = torch.randn(32, 10)

print("A shape:", A.shape)
print("B shape:", B.shape)
print("Output shape:", (A @ B).shape)


# 23. Practice Exercises

Try solving these before looking at the solutions.

## Exercise 1

Given:

$$
A.shape=(4,\ 3)
$$

and:

$$
B.shape=(3,\ 2)
$$

Is `A @ B` valid?

What is the output shape?

## Exercise 2

Can these matrices be multiplied?

$$
(5,\ 7) @ (6,\ 4)
$$

Explain why.

## Exercise 3

Predict:

$$
(32,\ 128) @ (128,\ 10)
$$

## Exercise 4

Calculate the dot product:

$$
\begin{array}{|c|c|c|}
\hline
1 & 2 & 3 \\
\hline
\end{array}
\cdot
\begin{array}{|c|c|c|}
\hline
4 & 5 & 6 \\
\hline
\end{array}
$$

## Exercise 5

Given:

$$
A.shape=(10,\ 5)
$$

and:

$$
B.shape=(20,\ 5)
$$

Which transpose allows multiplication?

What is the output shape?

## Exercise 6

For:

$$
A.shape=(8,\ 3,\ 4)
$$

and:

$$
B.shape=(8,\ 4,\ 6)
$$

predict the output of:

`torch.bmm(A, B)`

## Exercise 7

An input batch has shape:

$$
(64,\ 256)
$$

A linear layer has 100 output features.

What is the weight shape in `nn.Linear`?

What is the output shape?

## Exercise 8

If:

$$
X.shape=(32,\ 784)
$$

and:

$$
W.shape=(10,\ 784)
$$

write the correct manual multiplication.

## Exercise 9

Explain why `A * B` and `A @ B` are different.

## Exercise 10

Verify that:

`torch.matmul(A, B)`

and:

`A @ B`

produce the same result.


# 24. Shape Reasoning Challenges

Do not run code immediately.

## Challenge 1

$$
(7,\ 9) @ (9,\ 4)
\rightarrow ?
$$

## Challenge 2

$$
(16,\ 1,\ 32) @ (16,\ 32,\ 8)
\rightarrow ?
$$

## Challenge 3

$$
(5,\ 12) @ (3,\ 12)^T
\rightarrow ?
$$

## Challenge 4

A model receives:

$$
X.shape=(128,\ 512)
$$

and uses:

`nn.Linear(512, 64)`

What are:

- `weight.shape`
- `bias.shape`
- output shape

## Challenge 5

For:

$$
A.shape=(4,\ 6,\ 3)
$$

and:

$$
B.shape=(3,\ 10)
$$

what is the output shape of:

`torch.matmul(A, B)`?


# 25. Exercise Solutions


In [ ]:
# Exercise 1
A = torch.randn(4, 3)
B = torch.randn(3, 2)
print("Exercise 1:", (A @ B).shape)

# Exercise 2
print("Exercise 2: invalid because 7 != 6")

# Exercise 3
A = torch.randn(32, 128)
B = torch.randn(128, 10)
print("Exercise 3:", (A @ B).shape)

# Exercise 4
x = torch.tensor([1, 2, 3])
y = torch.tensor([4, 5, 6])
print("Exercise 4:", torch.dot(x, y))

# Exercise 5
A = torch.randn(10, 5)
B = torch.randn(20, 5)
print("Exercise 5:", (A @ B.T).shape)

# Exercise 6
A = torch.randn(8, 3, 4)
B = torch.randn(8, 4, 6)
print("Exercise 6:", torch.bmm(A, B).shape)

# Exercise 7
linear = torch.nn.Linear(256, 100)
X = torch.randn(64, 256)
print("Exercise 7 weight:", linear.weight.shape)
print("Exercise 7 output:", linear(X).shape)

# Exercise 8
X = torch.randn(32, 784)
W = torch.randn(10, 784)
print("Exercise 8:", (X @ W.T).shape)

# Exercise 10
A = torch.randn(3, 4)
B = torch.randn(4, 5)
print("Exercise 10:", torch.allclose(torch.matmul(A, B), A @ B))


# 26. Key Takeaways

In this notebook, we learned:

- Element-wise multiplication vs matrix multiplication
- Dot products
- Row-column multiplication
- Inner-dimension rule
- Output-shape reasoning
- `torch.matmul()`
- `@`
- Matrix-vector multiplication
- Matrix-matrix multiplication
- Batched matrix multiplication
- `torch.bmm()`
- Transpose
- Linear layers from first principles
- Neural-network weight shapes
- Common matrix multiplication errors

The most important rule is:

$$
(m,\ \boxed{n}) @ (\boxed{n},\ p)
\rightarrow
(m,\ p)
$$

> **Inner dimensions must match. Outer dimensions determine the output.**


# 27. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. What is the difference between `*` and `@`?
2. What is a dot product?
3. Why is matrix multiplication a collection of dot products?
4. What is the inner-dimension rule?
5. How do you predict the output shape?
6. What does `torch.matmul()` do?
7. Is `A @ B` always the same as `B @ A`?
8. What does transpose do?
9. Why is transpose commonly used with weight matrices?
10. What shape does `nn.Linear(128, 64).weight` have?
11. Why does the manual linear operation use `weight.T`?
12. What is batched matrix multiplication?
13. What does `torch.bmm()` expect?
14. What is the difference between `matmul()` and `bmm()`?
15. How does matrix multiplication appear inside a neural network?


# Next Notebook

# 06 — Autograd and Automatic Differentiation

In the next notebook, we will study:

- Derivatives from zero
- Gradients
- `requires_grad`
- Computational graphs
- `grad_fn`
- `backward()`
- Gradient accumulation
- `zero_grad()`
- `torch.no_grad()`
- `detach()`
- Chain rule intuition
- Manual gradients vs PyTorch gradients
- Why gradients matter in neural networks
- Common autograd mistakes
